# 03 — Feature Analysis
### Spectral, Temporal and Spatial EEG Dynamics Across Attention States

**Project:** EEG Attention State Detection (Focused / Unfocused / Drowsy)

This notebook does **not** train a classifier. Its only question is:

> **What actually changes in the EEG when the attention state changes?**

That matters because it gives notebook 04 (baselines) and notebook 05 (deep learning) a physiological/data-analysis
motivation instead of throwing raw signal at an arbitrary architecture. Concretely, this notebook:

1. loads the cleaned windows from notebook 02 (`windows.csv` + `continuous/*.npz`),
2. computes spectral (band power, absolute + relative), temporal (Hjorth, RMS, line length, ...) and ratio
   (theta/beta, ...) features, per channel and per window,
3. looks at class differences with plots, a topography sketch, Kruskal-Wallis tests, a correlation matrix and PCA,
4. saves one row per window to `data/processed/eeg_features.csv`.

### Inherited caveats (from notebook 02 — still true here)

- **Labels are protocol-derived**, not verified per participant (first ~10 min focused, next ~10 min unfocused,
  remainder drowsy). Treat class differences as differences between *protocol phases*, not confirmed mental states.
- **The 34 → 5 recordings-to-subjects mapping is still unresolved.** `group` = recording, so anything that looks
  like a "subject effect" below is really a *recording* effect (which may or may not correspond to a person).
  This is exactly why we also do a PCA colored by recording, not just by label.
- Windows from the same recording are not independent samples. The Kruskal-Wallis tests here are **exploratory**,
  not a substitute for a properly aggregated statistical test.

## 0. Imports

In [ ]:
import json
import itertools
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import welch
from scipy.stats import kruskal
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(x, **kwargs):
        return x

plt.rcParams["figure.figsize"] = (14, 4)
pd.set_option("display.max_columns", 60)

# numpy >= 2.0 renamed trapz -> trapezoid; this works on old and new numpy alike.
_trapz = getattr(np, "trapezoid", None) or np.trapz

## 1. Configuration

Loads the exact parameters notebook 02 used (`preprocessing_config.json`), so band edges, window length and the
active-channel list can never silently drift out of sync between notebooks.

In [ ]:
PROJECT_ROOT = Path("..").resolve()
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
CONT_DIR = PROCESSED_DIR / "continuous"
INTERIM_DIR = PROJECT_ROOT / "data" / "interim"

with open(PROCESSED_DIR / "preprocessing_config.json") as f:
    PP_CONFIG = json.load(f)

ACTIVE_CHANNELS = PP_CONFIG["active_channels"]
FS = PP_CONFIG["fs"]
WIN_SEC = PP_CONFIG["win_sec"]
LABEL_NAMES = {int(k): v for k, v in PP_CONFIG["label_names"].items()}
WIN_SAMPLES = int(round(WIN_SEC * FS))

print(f"Active channels ({len(ACTIVE_CHANNELS)}): {ACTIVE_CHANNELS}")
print(f"fs = {FS} Hz, window = {WIN_SEC}s = {WIN_SAMPLES} samples")
print(f"Labels: {LABEL_NAMES}")

# EEG frequency bands. 45 Hz upper edge matches the notebook-02 low-pass filter — band power above
# that frequency would just reflect the filter roll-off, not brain activity.
BANDS = {
    "delta": (1, 4),
    "theta": (4, 8),
    "alpha": (8, 13),
    "beta": (13, 30),
    "gamma": (30, 45),
}
TOTAL_BAND = (1, 45)

LABEL_ORDER = ["focused", "unfocused", "drowsy"]      # fixed order for consistent plots/tables
PALETTE = {"focused": "#1f77b4", "unfocused": "#ff7f0e", "drowsy": "#2ca02c"}

RANDOM_SEED = 42

## 2. Load kept windows and verify

`windows.csv` has one row per candidate window (kept and rejected). We only ever use `keep == True` rows here —
rejected windows (artifacts, flat channels, label-boundary straddling, edge effects, packet gaps) were already
screened out in notebook 02.

In [ ]:
windows = pd.read_csv(PROCESSED_DIR / "windows.csv")
windows = windows[windows["keep"]].reset_index(drop=True)
windows["label_name"] = windows["label"].map(LABEL_NAMES)

print(f"Kept windows: {len(windows):,}")
print(f"Recordings:   {windows['recording'].nunique()}")
print(f"Channels:     {len(ACTIVE_CHANNELS)}  ({ACTIVE_CHANNELS})")
print(f"Window length: {WIN_SEC}s")
print()
print("Class distribution:")
print(windows["label_name"].value_counts().reindex(LABEL_ORDER).to_string())
print()
print("Windows per recording (head):")
print(windows.groupby("recording").size().head().to_string())

In [ ]:
# Sanity check: every recording referenced in windows.csv must have a matching .npz, with the same channel order.
missing = [r for r in windows["recording"].unique() if not (CONT_DIR / Path(r).with_suffix(".npz").name).exists()]
assert not missing, f"Missing .npz for: {missing}"

ref_channels = None
for r in windows["recording"].unique():
    chans = list(np.load(CONT_DIR / Path(r).with_suffix(".npz").name)["channels"])
    if ref_channels is None:
        ref_channels = chans
    assert chans == ref_channels, f"{r} has channel order {chans}, expected {ref_channels}"
print("All recordings agree on channel order:", ref_channels)

## 3. Spectral features: absolute and relative band power

For every window we take the (already filtered, robust-scaled) segment and compute one Welch PSD per channel, then
integrate it over each band:

$$ P_{band} = \int_{f_1}^{f_2} PSD(f)\, df \qquad\qquad RelativePower_{band} = \frac{P_{band}}{P_{1-45\,Hz}} $$

**Relative power matters here**: because scaling in notebook 02 is per-recording, absolute power is not directly
comparable across recordings, but relative power (a *ratio* within the same window) mostly cancels that out.

We keep both, since exploratory work later may still want absolute levels, but relative power is the one to trust
for cross-recording comparisons.

Each recording is processed once: we gather all its kept windows into a single `(n_windows, win, n_channels)` array
and run `welch` vectorized over that whole batch (`axis=1`), rather than looping window by window in Python.

In [ ]:
def band_power_table(recording_name: str, win_starts: np.ndarray) -> pd.DataFrame:
    npz = np.load(CONT_DIR / Path(recording_name).with_suffix(".npz").name)
    signal = npz["signal"]                                    # (n_samples, n_channels)

    idx = win_starts[:, None] + np.arange(WIN_SAMPLES)[None, :]
    seg = signal[idx]                                          # (n_windows, win, n_channels)

    freqs, psd = welch(seg, fs=FS, nperseg=WIN_SAMPLES, axis=1)  # psd: (n_windows, n_freq, n_channels)

    lo, hi = TOTAL_BAND
    total_mask = (freqs >= lo) & (freqs < hi)
    total_power = _trapz(psd[:, total_mask, :], freqs[total_mask], axis=1)   # (n_windows, n_channels)
    total_power = np.maximum(total_power, 1e-12)                              # avoid /0 on a dead channel

    out = {}
    for band, (lo, hi) in BANDS.items():
        mask = (freqs >= lo) & (freqs < hi)
        bp = _trapz(psd[:, mask, :], freqs[mask], axis=1)     # (n_windows, n_channels)
        for c, ch in enumerate(ACTIVE_CHANNELS):
            out[f"{ch}_{band}_abs"] = bp[:, c]
            out[f"{ch}_{band}_rel"] = bp[:, c] / total_power[:, c]
    return pd.DataFrame(out), seg

In [ ]:
spectral_frames = []
seg_cache = {}     # keep the raw window segments too, section 7 (temporal features) reuses them

for rec, group in tqdm(windows.groupby("recording", sort=False), desc="Spectral features"):
    starts = group["start"].to_numpy()
    feats, seg = band_power_table(rec, starts)
    feats.index = group.index
    spectral_frames.append(feats)
    seg_cache[rec] = (group.index.to_numpy(), seg)

spectral = pd.concat(spectral_frames).sort_index()
print(f"Spectral features: {spectral.shape[1]} columns "
      f"({len(ACTIVE_CHANNELS)} channels x {len(BANDS)} bands x 2 [abs, rel])")
spectral.head()

## 4. Global (channel-averaged) band power

Per-channel features are useful for topography and for the model later, but for readable plots and statistics we
also want one number per band per window: the mean across channels.

In [ ]:
global_rel = pd.DataFrame({
    f"global_{band}": spectral[[f"{ch}_{band}_rel" for ch in ACTIVE_CHANNELS]].mean(axis=1)
    for band in BANDS
})
global_abs = pd.DataFrame({
    f"global_{band}_abs": spectral[[f"{ch}_{band}_abs" for ch in ACTIVE_CHANNELS]].mean(axis=1)
    for band in BANDS
})
global_feats = pd.concat([global_rel, global_abs], axis=1)
global_feats.head()

In [ ]:
plot_df = windows[["label_name"]].join(global_rel)

fig, axes = plt.subplots(1, len(BANDS), figsize=(4 * len(BANDS), 4), sharey=False)
for ax, band in zip(axes, BANDS):
    data_by_class = [plot_df.loc[plot_df.label_name == lab, f"global_{band}"] for lab in LABEL_ORDER]
    bp = ax.boxplot(data_by_class, labels=LABEL_ORDER, showfliers=False, patch_artist=True)
    for patch, lab in zip(bp["boxes"], LABEL_ORDER):
        patch.set_facecolor(PALETTE[lab])
        patch.set_alpha(0.6)
    ax.set_title(f"{band} (relative)")
    ax.tick_params(axis="x", rotation=30)
axes[0].set_ylabel("Relative power")
plt.suptitle("Relative band power by attention state (global, channel-averaged)")
plt.tight_layout()
plt.show()

> Read the boxplots before assuming a direction. A common expectation in drowsiness research is higher
> theta/delta and lower beta during drowsiness, but that is a hypothesis to check against *this* dataset and this
> protocol-derived labeling, not something to assume.

## 5. Band-power topography

A schematic (not anatomically exact) layout of the 14 Emotiv EPOC electrodes, colored by mean relative power per
class. This uses only the channels notebook 02 kept as active — if some electrodes were dropped, they will simply
be missing from the plot.

For a publication-quality scalp topomap (interpolated, anatomically correct) this could later be redone with
`mne.viz.plot_topomap`, using a standard montage. Here we keep it dependency-light.

In [ ]:
# Approximate 10-20-style positions (top-down view, nose points up), good enough for a schematic comparison
ELECTRODE_POS = {
    "AF3": (-0.25, 0.92), "AF4": (0.25, 0.92),
    "F7": (-0.85, 0.55), "F3": (-0.40, 0.55), "F4": (0.40, 0.55), "F8": (0.85, 0.55),
    "FC5": (-0.65, 0.20), "FC6": (0.65, 0.20),
    "T7": (-1.00, 0.00), "T8": (1.00, 0.00),
    "P7": (-0.85, -0.55), "P8": (0.85, -0.55),
    "O1": (-0.25, -0.92), "O2": (0.25, -0.92),
}
plotted_channels = [c for c in ACTIVE_CHANNELS if c in ELECTRODE_POS]
missing_pos = [c for c in ACTIVE_CHANNELS if c not in ELECTRODE_POS]
if missing_pos:
    print(f"No plotting position for: {missing_pos} (skipped in the topography plot only)")


def topography_grid(bands_to_show):
    fig, axes = plt.subplots(len(bands_to_show), len(LABEL_ORDER),
                              figsize=(4 * len(LABEL_ORDER), 4 * len(bands_to_show)))
    axes = np.atleast_2d(axes)

    # shared color scale per band row, so the three classes are visually comparable
    for r, band in enumerate(bands_to_show):
        cols = [f"{ch}_{band}_rel" for ch in plotted_channels]
        vmin = spectral[cols].min().min()
        vmax = spectral[cols].max().max()
        for c, lab in enumerate(LABEL_ORDER):
            ax = axes[r, c]
            mask = windows["label_name"] == lab
            vals = spectral.loc[mask, cols].mean(axis=0).values
            xs = [ELECTRODE_POS[ch][0] for ch in plotted_channels]
            ys = [ELECTRODE_POS[ch][1] for ch in plotted_channels]
            sca = ax.scatter(xs, ys, c=vals, s=1400, cmap="RdBu_r", vmin=vmin, vmax=vmax,
                              edgecolors="k", linewidths=0.5)
            for ch, x, y in zip(plotted_channels, xs, ys):
                ax.annotate(ch, (x, y), ha="center", va="center", fontsize=7)
            circle = plt.Circle((0, 0), 1.15, fill=False, lw=1)
            ax.add_patch(circle)
            ax.set_xlim(-1.4, 1.4); ax.set_ylim(-1.4, 1.4)
            ax.set_aspect("equal"); ax.axis("off")
            ax.set_title(f"{band} — {lab}" if r == 0 else lab)
            if c == len(LABEL_ORDER) - 1:
                plt.colorbar(sca, ax=ax, fraction=0.046, label="mean relative power")
    plt.tight_layout()
    plt.show()


topography_grid(["theta", "alpha", "beta"])

## 6. EEG ratios

Theta/beta ratio (TBR) and related ratios are common candidate markers of attention/vigilance in the EEG
literature. We compute them **globally first** (channel-averaged power), and only break them out per channel if the
global version already looks informative — adding 14x more near-duplicate columns before checking that is not
worth the redundancy it creates (see the correlation matrix in section 9).

In [ ]:
eps = 1e-12
ratios = pd.DataFrame({
    "theta_beta_ratio": global_rel["global_theta"] / (global_rel["global_beta"] + eps),
    "alpha_beta_ratio": global_rel["global_alpha"] / (global_rel["global_beta"] + eps),
    "theta_alpha_over_beta": (global_rel["global_theta"] + global_rel["global_alpha"])
                              / (global_rel["global_beta"] + eps),
})

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, ratios.columns):
    data_by_class = [ratios.loc[windows.label_name == lab, col] for lab in LABEL_ORDER]
    bp = ax.boxplot(data_by_class, labels=LABEL_ORDER, showfliers=False, patch_artist=True)
    for patch, lab in zip(bp["boxes"], LABEL_ORDER):
        patch.set_facecolor(PALETTE[lab]); patch.set_alpha(0.6)
    ax.set_title(col)
    ax.tick_params(axis="x", rotation=30)
plt.suptitle("Candidate EEG ratios by attention state (global)")
plt.tight_layout()
plt.show()

## 7. Temporal (time-domain) features

Since this project is about EEG *dynamics*, not just spectral content, we add a compact set of time-domain
features per channel, reusing the raw window segments already gathered in section 3 (`seg_cache`), so the signal is
not read from disk twice:

- **mean, std, RMS, peak-to-peak** — basic amplitude descriptors,
- **line length** — sum of absolute first differences, sensitive to how "busy" the signal is,
- **zero-crossing rate** — how often the (mean-centered) signal changes sign,
- **Hjorth activity, mobility, complexity** — the standard EEG time-domain triplet:

$$ Activity = Var(x) \qquad Mobility = \sqrt{\frac{Var(x')}{Var(x)}} \qquad Complexity = \frac{Mobility(x')}{Mobility(x)} $$

In [ ]:
def hjorth(seg):
    """seg: (n_windows, win, n_channels). Returns activity, mobility, complexity, each (n_windows, n_channels)."""
    d1 = np.diff(seg, axis=1)
    d2 = np.diff(d1, axis=1)

    var0 = seg.var(axis=1)
    var1 = d1.var(axis=1)
    var2 = d2.var(axis=1)

    activity = var0
    mobility = np.sqrt(np.maximum(var1, 0) / (var0 + 1e-12))
    mobility_d1 = np.sqrt(np.maximum(var2, 0) / (var1 + 1e-12))
    complexity = mobility_d1 / (mobility + 1e-12)
    return activity, mobility, complexity


def temporal_feature_table(seg):
    """seg: (n_windows, win, n_channels) -> DataFrame indexed 0..n_windows-1."""
    mean = seg.mean(axis=1)
    std = seg.std(axis=1)
    rms = np.sqrt((seg ** 2).mean(axis=1))
    ptp = seg.max(axis=1) - seg.min(axis=1)
    line_length = np.abs(np.diff(seg, axis=1)).sum(axis=1)
    centered = seg - mean[:, None, :]
    zcr = (np.diff(np.sign(centered), axis=1) != 0).mean(axis=1)
    activity, mobility, complexity = hjorth(seg)

    out = {}
    for c, ch in enumerate(ACTIVE_CHANNELS):
        out[f"{ch}_mean"] = mean[:, c]
        out[f"{ch}_std"] = std[:, c]
        out[f"{ch}_rms"] = rms[:, c]
        out[f"{ch}_ptp"] = ptp[:, c]
        out[f"{ch}_line_length"] = line_length[:, c]
        out[f"{ch}_zcr"] = zcr[:, c]
        out[f"{ch}_hjorth_activity"] = activity[:, c]
        out[f"{ch}_hjorth_mobility"] = mobility[:, c]
        out[f"{ch}_hjorth_complexity"] = complexity[:, c]
    return out

In [ ]:
temporal_frames = []
for rec, (row_idx, seg) in tqdm(seg_cache.items(), desc="Temporal features"):
    feats = pd.DataFrame(temporal_feature_table(seg), index=row_idx)
    temporal_frames.append(feats)

temporal = pd.concat(temporal_frames).sort_index()
print(f"Temporal features: {temporal.shape[1]} columns ({len(ACTIVE_CHANNELS)} channels x 9 descriptors)")
temporal.head()

In [ ]:
global_temporal = pd.DataFrame({
    "global_hjorth_mobility": temporal[[f"{c}_hjorth_mobility" for c in ACTIVE_CHANNELS]].mean(axis=1),
    "global_hjorth_complexity": temporal[[f"{c}_hjorth_complexity" for c in ACTIVE_CHANNELS]].mean(axis=1),
    "global_line_length": temporal[[f"{c}_line_length" for c in ACTIVE_CHANNELS]].mean(axis=1),
})

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, global_temporal.columns):
    data_by_class = [global_temporal.loc[windows.label_name == lab, col] for lab in LABEL_ORDER]
    bp = ax.boxplot(data_by_class, labels=LABEL_ORDER, showfliers=False, patch_artist=True)
    for patch, lab in zip(bp["boxes"], LABEL_ORDER):
        patch.set_facecolor(PALETTE[lab]); patch.set_alpha(0.6)
    ax.set_title(col)
    ax.tick_params(axis="x", rotation=30)
plt.suptitle("Global time-domain features by attention state")
plt.tight_layout()
plt.show()

## 8. Statistical comparison between states (exploratory)

For each **global** feature we run a Kruskal-Wallis test (a non-parametric alternative to one-way ANOVA — no
Gaussian assumption) across the three classes:

$$ H_0: D_{focused} = D_{unfocused} = D_{drowsy} $$

**Important caveat, stated up front rather than in a footnote:** each window is treated here as one observation,
but windows from the same recording overlap in time and are not independent. A large sample size (thousands of
windows) will make even small, non-meaningful differences look "significant". Read this table as *which features
separate the classes best in this sample*, not as confirmatory inference. A stricter version (one aggregated value
per recording, `n = 34`) is suggested below as a follow-up.

In [ ]:
global_all = pd.concat([global_rel, ratios, global_temporal], axis=1)

rows = []
for col in global_all.columns:
    groups = [global_all.loc[windows.label_name == lab, col].dropna() for lab in LABEL_ORDER]
    stat, p = kruskal(*groups)
    rows.append({"feature": col, "H": stat, "p": p})

stats_df = pd.DataFrame(rows).sort_values("H", ascending=False).reset_index(drop=True)
stats_df["significant_p<0.001"] = stats_df["p"] < 0.001
display(stats_df)

In [ ]:
top = stats_df.head(8)
fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(top["feature"][::-1], top["H"][::-1])
ax.set_xlabel("Kruskal-Wallis H statistic")
ax.set_title("Global features that separate focused/unfocused/drowsy most (this sample)")
plt.tight_layout()
plt.show()

**Suggested follow-up (not run here):** aggregate each feature to one value per recording (e.g. the median over
that recording's kept windows for a given label) and re-run Kruskal-Wallis with `n = 34` instead of `n = ` (number
of windows). That test would be slower to reach significance but far more defensible, and it sidesteps the
window-independence problem entirely.

## 9. Feature correlation

A correlation matrix of the **global** features tells us which ones are redundant before they go into a model —
for example, absolute and relative power for the same band are expected to correlate strongly with each other and
with a ratio that already uses that band.

In [ ]:
corr = global_all.corr()

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(corr.values, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr.columns))); ax.set_xticklabels(corr.columns, rotation=90, fontsize=8)
ax.set_yticks(range(len(corr.columns))); ax.set_yticklabels(corr.columns, fontsize=8)
plt.colorbar(im, ax=ax, label="Pearson r")
ax.set_title("Correlation between global features")
plt.tight_layout()
plt.show()

In [ ]:
pairs = corr.where(~np.eye(len(corr), dtype=bool)).abs().unstack().sort_values(ascending=False)
pairs = pairs[~pairs.index.duplicated()].head(10)
print("Most correlated feature pairs (|r|):")
for (a, b), r in pairs.items():
    print(f"  {a:28s} <-> {b:28s}  r = {corr.loc[a, b]:+.2f}")

## 10. PCA: is there attention-state structure, or is it dominated by recording differences?

Two PCA projections of the same (standardized) global feature set:

- colored by **attention state** — is there any visible structure at all?
- colored by **recording** — could that structure just be "which recording this window came from"?

Once the subject mapping (section 10 of notebook 02) is resolved, add a third plot colored by subject — the
comparison between "colored by recording" and "colored by subject" is what will actually tell us whether a
recording effect is a subject effect.

In [ ]:
X = StandardScaler().fit_transform(global_all.fillna(global_all.median()))
pca = PCA(n_components=2, random_state=RANDOM_SEED)
pcs = pca.fit_transform(X)
print(f"Explained variance ratio: PC1={pca.explained_variance_ratio_[0]:.1%}, "
      f"PC2={pca.explained_variance_ratio_[1]:.1%}")

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for lab in LABEL_ORDER:
    m = windows["label_name"] == lab
    axes[0].scatter(pcs[m, 0], pcs[m, 1], s=4, alpha=0.4, color=PALETTE[lab], label=lab)
axes[0].set_title("PCA colored by attention state")
axes[0].legend(markerscale=3)

recs = windows["recording"].astype("category")
sca = axes[1].scatter(pcs[:, 0], pcs[:, 1], s=4, alpha=0.4, c=recs.cat.codes, cmap="tab20")
axes[1].set_title("PCA colored by recording")
handles, _ = sca.legend_elements(num=min(10, recs.cat.categories.size))
axes[1].legend(handles, list(recs.cat.categories[:len(handles)]), fontsize=6, ncol=2,
               title="recording (truncated)", loc="best")

for ax in axes:
    ax.set_xlabel("PC1"); ax.set_ylabel("PC2")
plt.tight_layout()
plt.show()

Read this side by side, not in isolation. If the same regions light up in both plots — i.e. state and recording
are confounded in feature space — any classifier evaluated by randomly splitting windows will look better than it
really is, because it can partly learn "which recording" instead of "which attention state". That is precisely why
notebook 04 must evaluate by recording (or, once available, by subject), never by randomly splitting windows.

## 11. Save features

One row per kept window: identifiers, label, per-channel spectral (absolute + relative) and temporal features,
plus the global/ratio summary columns used above. Notebook 04 can use this directly, or engineer further from the
per-channel columns.

In [ ]:
features = pd.concat([
    windows[["recording", "rec_num", "start", "end", "label", "label_name", "group"]].reset_index(drop=True),
    spectral.reset_index(drop=True),
    temporal.reset_index(drop=True),
    global_rel.reset_index(drop=True),
    global_temporal.reset_index(drop=True),
    ratios.reset_index(drop=True),
], axis=1)

out_path = PROCESSED_DIR / "eeg_features.csv"
features.to_csv(out_path, index=False)
print(f"Saved {features.shape[0]:,} rows x {features.shape[1]} columns -> {out_path}")
features.head()

## 12. Summary and what's next

**What we found** (fill in once this has run on the real 34 recordings — do not assume the synthetic-data direction
carries over):
- [ ] Which bands/ratios separated the three states best (see the Kruskal-Wallis ranking in section 8)
- [ ] Whether the topography plots show a spatial pattern (e.g. frontal vs. posterior) or a diffuse one
- [ ] Whether PCA structure lines up with attention state, with recording, or with both at once

**Files written**

| File | Content |
|---|---|
| `data/processed/eeg_features.csv` | one row per kept window: identifiers, label, per-channel spectral + temporal features, global/ratio summaries |

**Carried-forward limitations**

- Labels are protocol-derived, not verified per participant.
- `group` is still the recording, not a verified subject — do not call any later evaluation "subject-independent"
  until notebook 02's subject mapping is resolved.
- The Kruskal-Wallis tests here are exploratory (non-independent windows); treat p-values as descriptive, not
  confirmatory.
- The topography plot is a schematic 2D layout, not an anatomically interpolated scalp map.

**Next:** `04_baseline_models.ipynb` — logistic regression / SVM / random forest (or similar) on these handcrafted
features, evaluated with a **recording-grouped** split (`GroupKFold` on `group`), to establish whether classical
features already work before trying representation learning on raw EEG in notebook 05.